# Entrenamiento final del mejor λ de la búsqueda fiel a v1 (100 épocas)

Reentrenamiento, con las 100 épocas completas, de la mejor configuración hallada en el notebook 24 (búsqueda de Optuna corregida, con una arquitectura de pérdida **idéntica a v1**: Dice + CE + clDice se combinan en un único módulo y ese módulo completo se envuelve en `DeepSupervisionWrapper`, de modo que clDice se recalcula en cada escala de deep supervision y no solo en resolución completa).

Ganador del estudio `cldice_lambda_only_search_v3_fiel_v1` (17 trials, único no dominado en las cuatro métricas a la vez):

```
weight_dice = 1.0
weight_ce = 1.0
lambda_cldice = 0.29695675878484407   (Trial 14)
skeleton_iterations = 10
```

A diferencia del notebook 22 (que reentrenaba el ganador del notebook 21, cuya arquitectura de búsqueda replicaba de forma no intencionada la de v2), este notebook utiliza el trainer corregido. Constituye, por tanto, la primera comparación en condiciones equivalentes con la implementación real de v1.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.09 GB


In [2]:
!python -m pip install -q nnunetv2 medpy scikit-image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")
NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"
CUSTOM_TRAINERS = BASE / "custom_trainers"
CUSTOM_TRAINERS.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)

DATASET_ID = 501
DATASET_FOLDER_NAME = "Dataset501_ImageCAS"
CONFIGURATION = "3d_fullres"
FOLD = 0

# Hiperparámetros del Trial 14 (notebook 24, arquitectura fiel a v1)
WEIGHT_DICE = 1.0
WEIGHT_CE = 1.0
LAMBDA_CLDICE = 0.29695675878484407
SKELETON_ITERATIONS = 10

CLASS_NAME = "nnUNetTrainerLambdaV1Fiel_100epochs"

GT_FOLDER = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / "gt_segmentations"

print("Config lista. Clase a entrenar:", CLASS_NAME)

Config lista. Clase a entrenar: nnUNetTrainerLambdaV1Fiel_100epochs


## 1. Funciones de métricas

Idénticas a las de los notebooks 19, 21, 22 y 23.


In [3]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)


def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing


def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)


def hard_cldice(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    return (2 * tprec * tsens) / (tprec + tsens + 1e-8)


def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n


def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n


def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)


def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)
    return {
        "dice": dice_score(pred, gt),
        "cldice": hard_cldice(pred, gt),
        "cobertura_principal": cobertura,
        "n_componentes": n_componentes,
        "hd95_crudo_mm": hd95_seguro(pred, gt, spacing),
    }


def evaluar_trainer(trainer_name):
    carpeta_val = (
        NNUNET_RESULTS / DATASET_FOLDER_NAME
        / f"{trainer_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}" / "validation"
    )
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    filas = []
    for pred_path in preds:
        case_id = pred_path.name.replace(".nii.gz", "")
        gt_path = GT_FOLDER / pred_path.name
        if not gt_path.exists():
            continue
        m = calcular_metricas_caso(pred_path, gt_path)
        m["case_id"] = case_id
        filas.append(m)

    df = pd.DataFrame(filas)
    resumen = {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }
    return df, resumen

print("Funciones de metricas cargadas.")

Funciones de metricas cargadas.


## 2. Generación del trainer con los hiperparámetros seleccionados, arquitectura fiel a v1 (100 épocas)


In [4]:
TRAINER_TEMPLATE = r'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_{epochs}epochs
)


def soft_erode_3d(x):
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)
    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


class SoftClDiceLoss(nn.Module):
    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()
        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)
        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):
    """Identico a v1 (notebook 12): combina Dice+CE+clDice en un unico modulo
    por escala; DeepSupervisionWrapper envuelve este modulo COMPLETO, asi que
    clDice se recalcula de forma independiente en cada escala de deep supervision."""

    def __init__(self, regional_loss, lambda_cldice=0.2, skeleton_iterations=10):
        super().__init__()
        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice
        self.soft_cldice = SoftClDiceLoss(iterations=skeleton_iterations, smooth=1e-5)

    def forward(self, logits, target):
        loss_regional = self.regional_loss(logits, target)

        probabilities = torch.softmax(logits.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]

        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)
        foreground_target = (target == 1).float()

        topology_loss = self.soft_cldice(foreground_probability, foreground_target)

        return loss_regional + self.lambda_cldice * topology_loss


class {class_name}(nnUNetTrainer_{epochs}epochs):

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.weight_dice = {weight_dice}
        self.weight_ce = {weight_ce}
        self.lambda_cldice = {lambda_cldice}
        self.skeleton_iterations = {skeleton_iterations}

    def on_train_start(self):
        super().on_train_start()
        self.print_to_log_file(
            f"[Mejor lambda, arquitectura fiel a v1, reentrenado 100 epochs] "
            f"weight_dice={{self.weight_dice}} weight_ce={{self.weight_ce}} "
            f"lambda_cldice={{self.lambda_cldice:.6f}} skeleton_iterations={{self.skeleton_iterations}}"
        )

    def _build_loss(self):
        regional_loss = DC_and_CE_loss(
            soft_dice_kwargs={{
                "batch_dice": self.configuration_manager.batch_dice,
                "smooth": 1e-5,
                "do_bg": False,
                "ddp": self.is_ddp
            }},
            ce_kwargs={{}},
            weight_ce=self.weight_ce,
            weight_dice=self.weight_dice,
            ignore_label=self.label_manager.ignore_label,
            dice_class=MemoryEfficientSoftDiceLoss
        )

                # 1. Dice + CE + clDice combinados en un único módulo (como en v1)
        loss = DiceCEPlusSoftClDice(
            regional_loss=regional_loss,
            lambda_cldice=self.lambda_cldice,
            skeleton_iterations=self.skeleton_iterations,
        )

                # 2. Módulo combinado completo envuelto en DeepSupervisionWrapper (como en v1)
        if self.enable_deep_supervision:
            deep_supervision_scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** i) for i in range(len(deep_supervision_scales))])
            weights[-1] = 0
            weights = weights / weights.sum()
            loss = DeepSupervisionWrapper(loss, weights)

        return loss
'''

codigo = TRAINER_TEMPLATE.format(
    class_name=CLASS_NAME,
    epochs=100,
    weight_dice=WEIGHT_DICE,
    weight_ce=WEIGHT_CE,
    lambda_cldice=LAMBDA_CLDICE,
    skeleton_iterations=SKELETON_ITERATIONS,
)
ruta_trainer = CUSTOM_TRAINERS / f"{CLASS_NAME}.py"
ruta_trainer.write_text(codigo)
print("Trainer escrito en:", ruta_trainer)

Trainer escrito en: /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerLambdaV1Fiel_100epochs.py


## 3. Entrenamiento (100 épocas completas)

Con ≈ 69 s/época (mismo diseño de pérdida que v1, con clDice en todas las escalas de deep supervision), 100 épocas equivalen a ≈ 115 min, más ≈ 16-20 min de validación, es decir, **≈ 2 h 10 min - 2 h 20 min** en A100.


In [5]:
!nnUNetv2_train \
    501 \
    3d_fullres \
    0 \
    -tr nnUNetTrainerLambdaV1Fiel_100epochs \
    --npz


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Trainer 'nnUNetTrainerLambdaV1Fiel_100epochs' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Searching for class nnUNetTrainerLambdaV1Fiel_100epochs in folder /content/drive/MyDrive/TFM_nnUNet/custom_trainers with current module None
  Inspecting module: nnUNetTrainerLambdaV1Fiel_100epochs
Found class nnUNetTrainerLambdaV1Fiel_100epochs in nnUNetTrainerLambdaV1Fiel_100epochs
Using trainer 'nnUNetTrainerLambdaV1Fiel_100epochs' from: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Using device: cuda:0

################################################################

## 4. Evaluación y guardado de métricas caso a caso (con case_id, para los tests estadísticos posteriores)


In [6]:
df_metricas, resumen = evaluar_trainer(CLASS_NAME)

print("=== Resultados: mejor lambda (arquitectura fiel a v1) reentrenado a 100 epocas ===")
print(f"Dice:          {resumen['dice_mean']:.4f}")
print(f"clDice:        {resumen['cldice_mean']:.4f}")
print(f"n_componentes: {resumen['n_componentes_mean']:.2f}")
print(f"HD95 crudo:    {resumen['hd95_crudo_mean']:.2f} mm")
print(f"n_casos:       {resumen['n_casos']}")

df_metricas.to_csv(BASE / "lambda_v1fiel_100epochs_metricas_por_caso.csv", index=False)
print("\nGuardado caso a caso en: lambda_v1fiel_100epochs_metricas_por_caso.csv")

=== Resultados: mejor lambda (arquitectura fiel a v1) reentrenado a 100 epocas ===
Dice:          0.7403
clDice:        0.8056
n_componentes: 16.15
HD95 crudo:    23.16 mm
n_casos:       40

Guardado caso a caso en: lambda_v1fiel_100epochs_metricas_por_caso.csv


In [7]:
# Comparación directa con los resultados conocidos (mismos 40 casos, 100 épocas)
comparativa = pd.DataFrame([
    {"modelo": "baseline",              "dice": 0.7459, "cldice": 0.8069, "n_componentes": 17.15, "hd95_mm": 24.02},
    {"modelo": "v1_lam02 (real)",       "dice": 0.7549, "cldice": 0.8221, "n_componentes": 14.72, "hd95_mm": 21.06},
    {"modelo": "v2_lam02",              "dice": 0.7482, "cldice": 0.8094, "n_componentes": 16.10, "hd95_mm": 23.35},
    {"modelo": "Optuna-Best (4 hiperp, arq. v2-like)", "dice": 0.7502, "cldice": 0.8113, "n_componentes": 16.80, "hd95_mm": 20.71},
    {"modelo": "Mejor lambda v1-fiel (aqui)", "dice": resumen["dice_mean"], "cldice": resumen["cldice_mean"],
     "n_componentes": resumen["n_componentes_mean"], "hd95_mm": resumen["hd95_crudo_mean"]},
])
print(comparativa.to_string(index=False))

                              modelo     dice   cldice  n_componentes   hd95_mm
                            baseline 0.745900 0.806900          17.15 24.020000
                     v1_lam02 (real) 0.754900 0.822100          14.72 21.060000
                            v2_lam02 0.748200 0.809400          16.10 23.350000
Optuna-Best (4 hiperp, arq. v2-like) 0.750200 0.811300          16.80 20.710000
         Mejor lambda v1-fiel (aqui) 0.740344 0.805624          16.15 23.155987


## 5. Siguiente paso: test estadístico pareado frente a v1

Al igual que en el notebook 23, se comparan caso a caso (Wilcoxon y t-test pareado) este modelo y `v1_lam02` y `baseline`, a partir de `comparacion_metricas_completo.csv` y del archivo `lambda_v1fiel_100epochs_metricas_por_caso.csv` generado en este notebook. Al reproducir ambos modelos la arquitectura real de v1, la comparación es equivalente y solo difiere el valor de λ (0.2970 frente a 0.2 en v1).
